# W6｜函数、调用与可见性

## 本周讨论什么？

- **场景一｜函数外的变量与同名参数，读到的会是谁？**
  - 给本次阈值加公共修正量，排查函数外变量与同名参数的混用。
- **场景二｜同时维护两组累计值，为什么会串到一起？**
  - 为不同实验分别累计，修复多个更新函数共用状态的问题。
- **场景三｜函数调用自己以后，每一层还记得原来的参数吗？**
  - 递归求和并记录进入、返回过程，核对每层参数与累计结果。
- **场景四｜第二轮结果，为什么多了第一轮的数据？**
  - 合并每轮的整数批次，让独立收集与主动追加各按需求工作。
- **场景五｜为新实验生成一份可独立修改的配置**
  - 生成可独立修改的实验配置，保留已有变量并读取当前公共设置。
- **场景六｜从同一个累加进度，分别继续计算**
  - 保存累加进度和历史，从同一进度分叉后分别继续计算。


## 如何使用

- **环境**：选择 `week-06/.venv` 对应的 Python kernel。
- **每部分顺序**：
  - 读需求与示例 → 运行输入格 → 先预测，再运行错误 Playground。
  - 在 Playground 修改代码、加 `print`，观察中间结果。
  - 将修复结果填入“函数组装”格 → 运行“单元检查”。
- **报错与重试**：
  - 初始代码有错、函数壳尚未实现，**初始检查失败是预期起点**；分部分运行，不必一键 Run All。
  - 重试时先运行本部分输入格，再运行修复后的代码；导入配置那一部分删掉导入语句后，也要恢复已经被覆盖的名字。
- **参考与协作**：
  - 参考解答默认折叠，是静态文字，不会自动执行。
  - 可以请 Agent 协助解释和修复；沿一次具体调用，核对它的解释。


## 场景一｜函数外的变量与同名参数，读到的会是谁？

### 场景

- 程序已有一个公共基准阈值，保存在函数外的 `threshold = 10` 中；`read_threshold()` 用来读取它。
- 现在需要临时指定一个阈值，再加上统一修正量 `offset = 2`，得到本次实际使用的阈值。例如临时指定 `4`，应算出 `6`，公共基准仍保持为 `10`。

### 需求

- 保留 `read_threshold()`：不接收参数，返回函数外公共变量 `threshold` 当前的值。
- 修复 `caller(threshold)`：接收本次临时阈值，返回 **传入的值 + 函数外当前的 `offset`**。
- 参数、公共阈值和修正量都是整数；调用函数不修改公共变量。

公共变量和函数参数都叫 `threshold`，但本次计算要使用的是传入的值。

### 具体输入输出

- 公共 `threshold=10`、`offset=2`：
  - `caller(4)` → `6`。
  - `caller(11)` → `13`。
  - `read_threshold()` → `10`。
- 只把公共 `threshold` 改为 `20`：`caller(4)` 仍为 `6`，`read_threshold()` 应变为 `20`。
- 再把 `offset` 改为 `3`：`caller(4)` 应变为 `7`。

### 你的任务

- 下方 `caller_trial` 的计算有误。运行前先预测它会返回什么，再用已有的打印观察：本次参数、修正量、读取函数的返回值分别是多少。
- 修复计算，使返回值满足需求；保留末尾检查结果的断言。调试时可以自行增删打印。
- 将修复后的函数体填入组装格的 `caller`，再运行单元检查。


In [ ]:
# 重新运行本格，恢复场景一的公共配置和本次输入。
threshold = 10
offset = 2
q1_argument_threshold = 4
q1_expected = 6

print("公共基准 threshold：", threshold)
print("统一修正量 offset：", offset)
print("本次传入的阈值：", q1_argument_threshold)
print("需求规定的结果：", q1_expected)

# 我的预测：当前程序会算出什么？为什么？


In [ ]:
# Playground：先预测，再运行；观察实际用了哪一份 threshold。
def read_threshold_trial():
    return threshold


def caller_trial(threshold):
    print("caller 的 threshold：", threshold)
    print("caller 的 offset：", offset)
    value = read_threshold_trial()
    print("读取函数返回：", value)
    return value + offset


q1_trial_result = caller_trial(q1_argument_threshold)
print("实际结果：", q1_trial_result)
assert q1_trial_result == q1_expected, "本次结果应为传入的阈值加上修正量"


In [ ]:
# 读取公共基准的函数保持不变；填入修复后的计算。
# 若保留调试用的读取调用，将 read_threshold_trial() 改为 read_threshold()。
def read_threshold():
    return threshold


def caller(threshold):
    raise NotImplementedError("请完成本次阈值的修正计算")


In [ ]:
# 每行：公共基准、修正量、本次参数、期望结果。
q1_cases = [
    (10, 2, 4, 6),
    (10, 2, 11, 13),
    (10, 2, 10, 12),
    (20, 2, 4, 6),
    (20, 3, 4, 7),
    (-3, -2, 0, -2),
    (10, 0, 4, 4),
]
q1_saved_threshold = threshold
q1_saved_offset = offset
try:
    for q1_public, q1_offset, q1_argument, q1_expected_result in q1_cases:
        threshold = q1_public
        offset = q1_offset
        assert read_threshold() == q1_public
        assert caller(q1_argument) == q1_expected_result, "应使用本次参数和当前修正量"
        assert threshold == q1_public, "调用不应修改公共 threshold"
        assert offset == q1_offset, "调用不应修改公共 offset"
finally:
    threshold = q1_saved_threshold
    offset = q1_saved_offset

print("场景一：全部检查通过")


<details>
<summary>参考解答（点击展开）</summary>

```python
def read_threshold():
    return threshold


def caller(threshold):
    return threshold + offset
```

#### 错误原因

- 原程序通过读取函数拿到公共基准 `10`，却把它用于本次阈值 `4` 的修正计算，因此得到 `10 + 2 = 12`，而不是需求中的 `4 + 2 = 6`。
- 调用 `read_threshold()` 不会把 `caller` 的同名参数传入它的环境。读取函数仍按自己的定义环境查找函数外的 `threshold`。

#### 修正前后的本质差别

- **修正前**：计算使用 `read_threshold() + offset`，取的是公共基准。
- **修正后**：计算使用 `threshold + offset`，在 `caller` 内取本次参数；`offset` 仍从函数外读取。
- 改变的是本次计算的数据来源。因此只改公共基准不影响计算结果，改变本次参数或公共修正量才会影响结果。

</details>


## 场景二｜同时维护两组累计值，为什么会串到一起？

### 场景

- 甲、乙两组实验的累计值分别从 `10`、`100` 开始。后续数据交替到达，每次只更新其中一组；误计时也可能扣回。
- 需要按起点列表创建各组的更新函数。以后调用哪一组的函数，就更新并返回哪一组的累计值，下一次从这个结果继续。

### 需求

- 实现 `make_counters(starts)`：接收整数列表，返回与起点逐项对应、长度相同的函数列表。起点可以重复，空输入返回 `[]`。
- 每个返回的函数支持 `advance(step=1)`：将本组当前值加上整数 `step`，返回更新后的整数，并记住它。`step` 可以为零或负数。
- 每个计数器独立保存状态，不使用模块全局累加器：
  - 同一次创建的不同位置互不影响，即使起点相同。
  - 不同次 `make_counters` 调用互不影响。
- 不修改 `starts`；创建后再修改起点列表，也不影响已创建的计数器。

### 具体输入输出

```python
a, b = make_counters([10, 100])
a()       # 11
b(2)      # 102
a(3)      # 14
b(0)      # 102

c = make_counters([10])[0]
c()       # 11，从自己的起点开始
a(-1)     # 13
```

- 起点为 `[1, 1]` 时，先后调用第一个计数器两次得到 `2`、`3`，再调用第二个计数器一次仍应得到 `2`。

### 你的任务

- 下方 `make_counters_trial` 的累计结果有误。先预测四次操作的返回值，再运行并与需求比较。
- 检查各个更新函数使用的是哪一份 `current`，修复实现；保留末尾检查结果的断言。调试时可以自行增删打印。
- 将修复后的函数体填入组装格的 `make_counters`，再运行单元检查。


### 补充知识

#### 函数的返回值也可以是函数

- 函数除了返回数字、列表，也可以把另一个函数作为结果返回，交给我们以后调用。

```python
def make_doubler():
    def double(x):
        return x * 2
    return double

f = make_doubler()  # 得到一个函数，保存在 f 中
f(3)                # 调用这个函数，得到 6
```

- `return double` 没有括号，返回的是函数本身；`f(3)` 有括号，才执行计算。
- 本场景返回的是一个**函数列表**。写下 `a, b = make_counters([10, 100])` 后，`a`、`b` 就是两个可以分别调用的更新函数。

#### `nonlocal` 关键字

- 返回的内层函数仍然引用外层的变量，这种关系称为**闭包**。外层函数调用结束后，被引用的变量仍然可用。
- `nonlocal current` 表示：下面对 `current` 的赋值，要更新**外层函数中已有的那个变量**。

```python
def make_counter():
    current = 10

    def advance(step=1):
        nonlocal current
        current += step
        return current

    return advance

a = make_counter()  # 外层调用结束，a 仍引用那份 current
a()                 # 11
a(3)                # 14，继续使用更新后的值
```

- 如果内层只读取 `current`，例如 `return current`，就不需要 `nonlocal`；这里使用它，是为了给外层变量重新赋值。
- **状态能保留，不代表各函数的状态独立。** 场景二还需要检查：不同更新函数是否引用了同一个 `current`。


In [ ]:
# 重新运行本格，恢复题 2 的样例；计数器在下一格重新建立。
q2_starts = [10, 100]
q2_steps = [(0, 1), (1, 2), (0, 3), (1, 0)]
q2_expected = [11, 102, 14, 102]
print("起点：", q2_starts)
print("操作（计数器位置，增量）：", q2_steps)
print("预期返回值：", q2_expected)

# 运行下一格前，先写下对错误代码的预测（输出、异常或被修改的数据）：
# 我的预测：


In [ ]:
# Playground：观察每个函数实际使用哪份 current。
def make_counters_trial(starts):
    counters = []
    for start in starts:
        current = start

        def advance(step=1):
            nonlocal current
            current += step
            return current

        counters.append(advance)
    return counters


q2_trial_starts_before = q2_starts.copy()
q2_trial_counters = make_counters_trial(q2_starts)
q2_trial_results = []
for q2_index, q2_step in q2_steps:
    q2_trial_value = q2_trial_counters[q2_index](q2_step)
    q2_trial_results.append(q2_trial_value)
    print("计数器位置、增量、实际返回值：", q2_index, q2_step, q2_trial_value)
print("实际返回值列表：", q2_trial_results)
print("原起点：", q2_starts)
assert q2_trial_results == q2_expected, "各计数器应从自己的起点独立累计"
assert q2_starts == q2_trial_starts_before, "创建和更新计数器不应修改起点列表"


In [ ]:
# 返回的每个函数都应支持 advance(step=1)。
def make_counters(starts):
    raise NotImplementedError("请完成 make_counters")


In [ ]:
q2_test_starts = [10, 100]
q2_before = q2_test_starts.copy()
q2_a, q2_b = make_counters(q2_test_starts)
assert q2_a() == 11
assert q2_b(2) == 102
assert q2_a(3) == 14
assert q2_b(0) == 102
assert q2_test_starts == q2_before

q2_c = make_counters([10])[0]
assert q2_c() == 11
assert q2_a(-1) == 13
assert q2_b() == 103
assert q2_c(0) == 11

q2_same_a, q2_same_b = make_counters([1, 1])
assert q2_same_a() == 2
assert q2_same_a() == 3
assert q2_same_b() == 2
assert q2_same_b(-2) == 0
assert make_counters([]) == []

q2_changed_starts = [2]
q2_saved_counter = make_counters(q2_changed_starts)[0]
q2_changed_starts[0] = 99
assert q2_saved_counter() == 3
print("题 2：全部检查通过")


<details>
<summary>参考解答（点击展开）</summary>

```python
def make_counters_trial(starts):
    def make_one(start):
        current = start

        def advance(step=1):
            nonlocal current
            current += step
            return current

        return advance

    counters = []
    for start in starts:
        new_counter = make_one(start)
        counters.append(new_counter)
    return counters
```

#### 错误原因

- 一次 `make_counters_trial` 调用中，循环反复给同一个 `current` 赋值；`for` 不会为每轮创建新的函数作用域。
- 返回的多个 `advance` 都引用这一份变量。循环结束时它是 `100`，甲加 `1` 后变为 `101`，乙再加 `2` 就从这里继续得到 `103`。

#### 修正前后的本质差别

- **修正前**：在循环中直接定义 `advance`，多个函数共同引用一次外层调用里的 `current`。
- **修正后**：每轮调用 `make_one(start)`，每次调用都建立自己的 `current`，返回的 `advance` 引用各自那一份变量。
- `nonlocal` 的含义没有改变，仍更新最近一层外层函数中已有的 `current`。真正起作用的是：每个计数器来自一次独立的 `make_one` 调用。

填入组装格时，用上面的函数体替换正式入口 `make_counters` 的函数体即可。

</details>


## 场景三｜函数调用自己以后，每一层还记得原来的参数吗？

### 场景

- 编写一个带执行日志的求和小程序：输入非负整数 `x`，用递归计算 `x + (x - 1) + … + 0`。
- 输出总和的同时，保留每一层的进入参数和返回结果，便于逐层核对计算过程。例如输入 `2`，总和应为 `3`，日志应记录处理 `2`、`1`、`0` 的过程。

### 需求

- 实现 `sum_trace(x)`，输入是 `0～20` 的整数，返回字典：
  - `total`：本次计算的整数总和。
  - `events`：按实际执行顺序记录事件的新列表。
- 每次递归调用开始时记录 `("enter", x)`，返回前记录 `("leave", x, subtotal)`：
  - `x` 是这一层收到的参数，`subtotal` 是这一层返回的累计和。
  - `x == 0` 时也记录进入和返回；其余情况递归计算子问题。
- 日志对应真实的递归过程，不使用闭式公式另造模拟轨迹。每次 `sum_trace` 的事件列表独立，后一次调用不改变前一次的结果。

### 具体输入输出

```python
sum_trace(2)
# {
#   "total": 3,
#   "events": [
#       ("enter", 2), ("enter", 1), ("enter", 0),
#       ("leave", 0, 0), ("leave", 1, 1), ("leave", 2, 3)
#   ]
# }

sum_trace(0)
# {"total": 0, "events": [("enter", 0), ("leave", 0, 0)]}
```

### 你的任务

- 下方 `sum_trace_trial` 的计算和日志有误。先预测输入 `2` 时的结果，再运行并逐条比较实际日志与需求。
- 检查每层用于累计的 `x` 和传给下一层的参数，修复实现；保留末尾检查结果的断言。调试时可以自行增删打印。
- 将修复后的函数体填入组装格的 `sum_trace`，再运行单元检查。


In [ ]:
# 重新运行本格，恢复题 3 的输入。
q3_x = 2
q3_expected = {
    "total": 3,
    "events": [
        ("enter", 2), ("enter", 1), ("enter", 0),
        ("leave", 0, 0), ("leave", 1, 1), ("leave", 2, 3),
    ],
}
print("输入 x：", q3_x)
print("预期：", q3_expected)

# 运行下一格前，先写下对错误代码的预测（输出、异常或被修改的数据）：
# 我的预测：


In [ ]:
# Playground：追踪每次 visit 收到的 x，以及返回前它变成了什么。
def sum_trace_trial(x):
    events = []

    def visit(x):
        events.append(("enter", x))
        if x == 0:
            subtotal = 0
        else:
            x -= 1
            subtotal = x + visit(x)
        events.append(("leave", x, subtotal))
        return subtotal

    total = visit(x)
    return {"total": total, "events": events}


q3_trial_result = sum_trace_trial(q3_x)
print("累计和：", q3_trial_result["total"])
for q3_event in q3_trial_result["events"]:
    print(q3_event)
assert q3_trial_result == q3_expected, "累计和与进入、返回轨迹都应符合需求"


In [ ]:
def sum_trace(x):
    raise NotImplementedError("请完成 sum_trace")


In [ ]:
q3_expected_two = {
    "total": 3,
    "events": [
        ("enter", 2), ("enter", 1), ("enter", 0),
        ("leave", 0, 0), ("leave", 1, 1), ("leave", 2, 3),
    ],
}
q3_first = sum_trace(2)
assert q3_first == q3_expected_two
q3_second = sum_trace(0)
assert q3_second == {"total": 0, "events": [("enter", 0), ("leave", 0, 0)]}
assert q3_first == q3_expected_two

# 用规范单独构造预期顺序：先依次进入，再逐层返回。
for q3_n in [1, 3, 20]:
    q3_result = sum_trace(q3_n)
    q3_expected_events = [("enter", level) for level in range(q3_n, -1, -1)]
    q3_expected_events += [
        ("leave", level, sum(range(level + 1)))
        for level in range(q3_n + 1)
    ]
    assert q3_result == {
        "total": sum(range(q3_n + 1)),
        "events": q3_expected_events,
    }

q3_first["events"].append(("edited",))
assert q3_second == {"total": 0, "events": [("enter", 0), ("leave", 0, 0)]}
assert sum_trace(2) == q3_expected_two
print("题 3：全部检查通过")


<details>
<summary>参考解答（点击展开）</summary>

```python
def sum_trace(x):
    events = []

    def visit(x):
        events.append(("enter", x))
        if x == 0:
            subtotal = 0
        else:
            subtotal = x + visit(x - 1)
        events.append(("leave", x, subtotal))
        return subtotal

    total = visit(x)
    return {"total": total, "events": events}
```

#### 错误原因

- 原程序先执行 `x -= 1`，改掉了当前这一层的参数；随后计算加数和记录返回日志时，用的都是已经减过的值。
- 因此总和漏掉了本层原本应加的数，进入与返回记录里的参数也对不上。这是当前层自己重新赋值造成的，不是下一层调用改掉了上一层的变量。

#### 修正前后的本质差别

- **修正前**：先修改本层的 `x`，再用修改后的值执行 `x + visit(x)`。
- **修正后**：保留本层的 `x`，执行 `x + visit(x - 1)`，只把 `x - 1` 的结果作为下一层的参数。
- 各层保留自己的参数，返回时仍能记录本层原始的 `x`。一次求和中的各层共同追加到同一份 `events`；下一次 `sum_trace` 调用会新建日志列表。

</details>


## 场景四｜第二轮结果，为什么多了第一轮的数据？

### 补充知识

#### 缺省输入与显式传入

```python
def scale(value, factor=2):  # value 在前，factor 在后；factor 的默认值是 2
    return value * factor

scale(3)            # 6：省略 factor，使用默认值 2
scale(3, 4)         # 12：按位置显式传入 factor=4
scale(3, factor=4)  # 12：按名称显式传入 factor=4
```

### 场景

- 每轮实验产生几批整数，需要合并为这一轮的结果。第一轮的 `[[1, 2], [], [3]]` 合并成 `[1, 2, 3]`，第二轮的 `[[9]]` 合并成 `[9]`；两轮结果要分别保存。
- 有时需要连续积累：主动提供已有的 `[100]`，把 `[[4], [5, 6]]` 追加进去，继续使用同一个结果列表。

### 需求

- 实现 `collect_batches(batches, target=None)`。`batches` 是整数列表组成的列表，各批可为空；按批次和批内顺序展开，保留重复项，不修改各批原始输入。
  - 按位置传参时，`batches` 是第一个参数，`target` 是第二个、也是最后一个参数；也可明确写为 `target=已有列表`。
- `target` 为 `None`（省略或显式传入）时，每次返回独立的新列表。修改结果不能改动原始批次或其他调用的结果。
- 显式传入 `target` 列表时（包括空列表），保留它原有的全部内容，再按批次与批内顺序把新元素追加到末尾，返回这个对象本身。给定的 `target` 不与任何原始批次共用对象。
- `batches` 为空时，仍按上述规则返回新空列表，或原样返回给定的 `target`。

### 具体输入输出

```python
first = collect_batches([[1, 2], [], [3]])   # [1, 2, 3]
second = collect_batches([[9]])             # [9]；first 仍为 [1, 2, 3]
# first 与 second 是不同的列表。

target = [100]
result = collect_batches([[4], [5, 6]], target)
# 原有的 100 保持在前，新元素 4、5、6 依次追加到后面。
# target 变为 [100, 4, 5, 6]，且 result is target。
```

### 你的任务

- 下方 `collect_batches_trial` 的两次默认调用会混合数据。运行前先预测两次返回的内容，再用已有打印观察第一次结果是否也发生了变化。
- 修复函数，让默认调用分别收集，显式给定列表时连续积累；保留末尾检查实际结果的断言。调试时可以自行增删打印。
- 将修复后的函数体填入组装格的 `collect_batches`，再运行单元检查。


In [ ]:
# 重新运行本格，恢复场景四的批次与给定列表。
q4_batches = [[1, 2], [], [3]]
q4_target = [100]
print("第一轮批次：", q4_batches)
print("第二轮批次：", [[9]])
print("连续积累时给定的 target：", q4_target)
print("预期两轮分别为 [1, 2, 3]、[9]；给定 target 追加后为 [100, 4, 5, 6]。")

# 我的预测：两次默认调用各返回什么？第一次结果还会改变吗？


In [ ]:
# Playground：先预测，再运行；观察默认调用和给定 target 的结果。
def collect_batches_trial(batches, target=[]):
    if target is None:
        target = []
    for batch in batches:
        target.extend(batch)
    return target


q4_first_trial = collect_batches_trial(q4_batches)
print("第一次返回：", q4_first_trial)
q4_second_trial = collect_batches_trial([[9]])
print("第二次返回：", q4_second_trial)
print("回头看第一次返回：", q4_first_trial)
print("两次返回是同一个对象吗？", q4_first_trial is q4_second_trial)
q4_given_trial = collect_batches_trial([[4], [5, 6]], q4_target)
print("给定 target 后的结果：", q4_given_trial)
print("返回的是给定的 target 吗？", q4_given_trial is q4_target)
print("原始批次：", q4_batches)

assert q4_first_trial == [1, 2, 3], "第一轮结果应保持为第一轮的数据"
assert q4_second_trial == [9], "第二轮应只收集本轮数据"
assert q4_first_trial is not q4_second_trial, "默认调用应返回不同的列表"
assert q4_given_trial is q4_target and q4_target == [100, 4, 5, 6]
assert q4_batches == [[1, 2], [], [3]], "收集不能修改原始批次"


In [ ]:
# 将修正后的过程组装进正式接口；不改下面的验收预期。
def collect_batches(batches, target=None):
    raise NotImplementedError("请把 Playground 中修正后的过程组装进来")


In [ ]:
q4_test_batches = [[1, 2], [], [3]]
q4_first = collect_batches(q4_test_batches)
q4_second = collect_batches([[9]])
assert q4_first == [1, 2, 3]
assert q4_second == [9]
assert q4_first is not q4_second

q4_single_batch = [[7]]
q4_none = collect_batches(q4_single_batch, None)
assert q4_none == [7] and q4_none is not q4_first
q4_none.append(8)
assert q4_single_batch == [[7]]
assert q4_test_batches == [[1, 2], [], [3]]
assert q4_first == [1, 2, 3]

q4_given = [100]
q4_result = collect_batches([[4], [5, 6]], q4_given)
assert q4_result is q4_given
assert q4_given == [100, 4, 5, 6]
q4_empty_target = []
assert collect_batches([[2, 2]], q4_empty_target) is q4_empty_target
assert q4_empty_target == [2, 2]
assert collect_batches([], q4_given) is q4_given
assert q4_given == [100, 4, 5, 6]
q4_empty_a, q4_empty_b = collect_batches([]), collect_batches([])
assert q4_empty_a == q4_empty_b == [] and q4_empty_a is not q4_empty_b
print("题 4：PASS（内容、输入保持、显式共享与默认独立）")


<details>
<summary>参考解答（点击展开）</summary>

```python
def collect_batches(batches, target=None):
    if target is None:
        target = []
    for batch in batches:
        target.extend(batch)
    return target
```

#### 错误原因

默认参数 `target=[]` 中的列表在函数定义时就已创建。省略 `target` 的多次调用会共用它，所以第二轮追加 `9` 后，第一轮返回的列表也跟着改变。原来的 `if target is None` 不能解决这个问题，因为省略参数时拿到的是默认列表，并不是 `None`。

#### 修正前后的本质差别

- **修正前**：省略 `target` 时，使用函数定义时创建的同一个列表，数据会跨调用积累。
- **修正后**：默认值改为 `None`，省略或显式传入 `None` 时，才在本次调用中创建新列表；不同调用各有自己的结果。显式传入列表时，仍直接向该对象追加并返回它，包括传入 `[]` 的情况。

</details>


## 场景五｜为新实验生成一份可独立修改的配置

### 场景

- `defaults.py` 保存公共配置：`seed = 42`、`limit = 3`、`labels = ["base"]`。当前程序另有 `seed = 7`，记录前一轮实验的种子。
- 现在为新实验指定种子，生成配置字典。每份配置都能追加自己的标签；公共处理上限改变后，新配置应采用新上限，已有配置仍保留原值。

### 需求

- 实现 `make_run_config(requested_seed)`，参数为整数；每次返回新字典，包含：
  - `seed`：本次传入的种子。
  - `limit`：`defaults.limit` 在本次调用时的值。
  - `labels`：从 `defaults.labels` 当前内容复制出的新列表。
- 导入和调用均不覆盖当前程序已有的 `seed = 7`，也不修改 `defaults` 中的数据。允许调整导入方式和函数实现。
- 修改返回配置的标签，不影响公共标签或其他配置；公共上限后来改变，不影响已生成配置中的上限。

### 具体输入输出

```python
first = make_run_config(11)
# {"seed": 11, "limit": 3, "labels": ["base"]}
first["labels"].append("trial")
second = make_run_config(12)
# {"seed": 12, "limit": 3, "labels": ["base"]}
# seed 仍为 7；defaults.seed 仍为 42；defaults.labels 仍为 ["base"]。

defaults.limit = 5
third = make_run_config(-1)
# {"seed": -1, "limit": 5, "labels": ["base"]}
# first 和 second 中的 limit 仍为 3。
```

### 你的任务

- 下方代码使用 `from defaults import *`。运行前先预测：导入后的 `seed`、追加标签后的公共标签，以及调整公共上限后新配置的 `limit` 各是什么。
- 根据实际打印修复导入方式和配置生成函数，保留末尾检查实际结果的断言。
- 修复后先重跑输入格，再运行修复后的 Playground，恢复先前被覆盖的名字和改动的数据；不要在修复代码中重设 `seed` 来掩盖覆盖问题。
- 将修复后的导入方式与函数一起填入组装格的 `make_run_config`，再运行单元检查。


In [ ]:
import defaults

# 重新运行本格，恢复场景五的公共配置和当前程序的 seed。
# 修复 Playground 后，也要先运行本格，再运行修复代码。
defaults.seed = 42
defaults.limit = 3
defaults.labels = ["base"]
seed = 7
q5_requested_seed = 11
print("当前程序已有的 seed：", seed)
print("默认配置：", defaults.seed, defaults.limit, defaults.labels)
print("本次传入的种子：", q5_requested_seed)

# 我的预测：导入、追加标签和调整公共上限后，相关数据分别是什么？


In [ ]:
# Playground：先预测，再运行；观察导入、标签修改与公共上限变化。
from defaults import *


def make_run_config_trial(requested_seed):
    return {"seed": requested_seed, "limit": limit, "labels": labels}


print("导入后，当前程序的 seed：", seed)
q5_first_trial = make_run_config_trial(q5_requested_seed)
q5_first_trial["labels"].append("trial")
print("追加标签后的第一份配置：", q5_first_trial)
print("公共 labels：", defaults.labels)
defaults.limit = 5
q5_second_trial = make_run_config_trial(12)
print("公共 limit 已调整为：", defaults.limit)
print("调整后生成的第二份配置：", q5_second_trial)
print("回头看第一份配置的 limit：", q5_first_trial["limit"])

assert seed == 7, "导入不应覆盖当前程序已有的 seed"
assert q5_first_trial == {"seed": 11, "limit": 3, "labels": ["base", "trial"]}
assert q5_second_trial == {"seed": 12, "limit": 5, "labels": ["base"]}
assert defaults.seed == 42 and defaults.labels == ["base"], "返回配置的标签应独立"
assert q5_first_trial is not q5_second_trial
assert q5_first_trial["labels"] is not q5_second_trial["labels"]
# 再次尝试时，先运行输入格恢复数据与 seed。


In [ ]:
# 将修复后的导入方式和函数一起组装到这一格。
# 此处不重设 seed；修复后先重跑输入格，清除先前错误留下的状态。
def make_run_config(requested_seed):
    raise NotImplementedError("请组装修复后的配置生成函数")


In [ ]:
import defaults

q5_saved_defaults = (defaults.seed, defaults.limit, defaults.labels.copy())
q5_saved_seed = seed
try:
    # 这一步检查导入是否已经覆盖了本模块的名字。
    assert seed == 7, "本模块 seed 应保持为 7；修复后请重跑输入格再验收"
    defaults.seed, defaults.limit, defaults.labels = 42, 3, ["base"]
    q5_first = make_run_config(11)
    assert q5_first == {"seed": 11, "limit": 3, "labels": ["base"]}
    q5_first["labels"].append("trial")
    q5_second = make_run_config(12)
    assert q5_second == {"seed": 12, "limit": 3, "labels": ["base"]}
    assert q5_first is not q5_second
    assert q5_first["labels"] is not q5_second["labels"]
    assert defaults.labels == ["base"]

    defaults.limit = 5
    q5_third = make_run_config(-1)
    assert q5_third == {"seed": -1, "limit": 5, "labels": ["base"]}
    assert q5_first["limit"] == q5_second["limit"] == 3
    assert seed == 7 and defaults.seed == 42
    assert defaults.labels == ["base"]
    print("题 5：PASS（名字保持、动态读取、返回配置隔离）")
finally:
    defaults.seed, defaults.limit, defaults.labels = q5_saved_defaults
    seed = q5_saved_seed


<details>
<summary>参考解答（点击展开）</summary>

两种写法任选一种。

**写法一：函数外导入**

```python
import defaults

def make_run_config(requested_seed):
    return {
        "seed": requested_seed,
        "limit": defaults.limit,
        "labels": defaults.labels.copy(),
    }
```

**写法二：函数内导入**

```python
def make_run_config(requested_seed):
    import defaults

    return {
        "seed": requested_seed,
        "limit": defaults.limit,
        "labels": defaults.labels.copy(),
    }
```

#### 错误原因

- **名字被覆盖**：`from defaults import *` 将配置名带入当前程序，覆盖了已有的 `seed`。
- **仍读到旧上限**：导入的 `limit` 绑定了当时的值；后来给 `defaults.limit` 赋新值，不会同步更新这个名字。
- **标签共用对象**：导入的 `labels` 与 `defaults.labels` 指向同一个列表，修改返回配置里的标签，也改到了公共配置。

#### 修正前后的本质差别

- **修正前**：直接使用导入的名字。
  - 公共 `seed` 覆盖当前程序的同名变量。
  - `limit` 停留在导入时绑定的值。
  - 多份返回配置共用公共标签列表。
- **修正后**：用模块名明确来源，并分开处理读取和复制。
  - `import defaults` 保留模块名，当前程序已有的 `seed` 保持不变。
  - 每次调用读取当前的 `defaults.limit`；已生成配置中的上限仍保留原值。
  - `defaults.labels.copy()` 为每份配置建立独立标签列表。

- **导入位置的差别**：第一种写法把模块名 `defaults` 放在函数外；第二种写法在调用时把这个名字绑定在函数内部。
  - 局部的是名字的作用域，并不意味着配置数据也自动变成每次调用独有。
  - 普通导入通常复用缓存中的同一个模块，所以仍需复制 `labels`，避免不同配置共用标签列表。

修复后先重跑输入格，恢复 `seed` 与公共配置，再运行修复后的导入和函数。修改源代码不会自动撤销先前运行造成的覆盖和数据改动。

</details>


## 场景六｜从同一个累加进度，分别继续计算

### 场景

- 你在逐批累加整数，记录每次更新前的值、增量和更新后的值。从 `10` 加 `2` 到 `12` 后，需要保存一份快照，再从这个进度分出另一条路线。
- 原路线继续加 `3`，新路线减 `5`，应分别得到 `15` 和 `7`。两条路线保留共同的前一条记录，随后各记各的；旧快照仍只记录保存时的进度，供单独查看和编辑。

### 需求

- 实现 `make_accumulator(start=0)`，返回含 `add`、`snapshot`、`fork` 三个可调用项的字典。`start` 和 `step` 都是整数，允许零和负数；初始值为 `start`，历史为空。
- 三个操作协作使用本实例的同一份当前值和历史：
  - `add(step)` 更新当前值并返回新整数；每次都追加 `{"before": 原值, "step": 增量, "after": 新值}`，增量为零也要记录。
  - `snapshot()` 返回 `{"value": 当前值, "history": 历史列表}`，不改变内部状态。返回的数据可独立修改，包括历史中的字典；后续更新也不能改变旧快照。
  - `fork()` 返回具有相同接口的新累加器，继承此刻的值和全部历史，不追加记录、不改变原实例。此后的父子分支互不影响，再次分叉也遵守同样规则。
- 不同调用建立的实例、各个分支及各份快照相互独立。使用函数与闭包组织，不引入 `class` 或模块全局累加器。

### 具体输入输出

```python
a = make_accumulator(10)
a["add"](2)                 # 12
saved = a["snapshot"]()
b = a["fork"]()
a["add"](3)                 # 15
b["add"](-5)                # 7

# a["snapshot"]()：
# {"value": 15, "history": [
#     {"before": 10, "step": 2, "after": 12},
#     {"before": 12, "step": 3, "after": 15}
# ]}
# b["snapshot"]()：
# {"value": 7, "history": [
#     {"before": 10, "step": 2, "after": 12},
#     {"before": 12, "step": -5, "after": 7}
# ]}
# saved 仍为 {"value": 12, "history": [{"before": 10, "step": 2, "after": 12}]}。
```

修改 `saved["history"][0]["after"]` 或向其历史追加记录，不改变 `a`、`b` 或其他快照。重新调用 `make_accumulator(10)`，仍得到值为 `10`、历史为空的新实例。

### 你的任务

- 下方 `make_accumulator_trial` 的历史没有正确隔离。先预测两条路线和旧快照的内容，再运行打印；继续观察修改旧快照中的字典后，其他数据是否改变。
- 修复快照和分叉，让实际值与完整历史都满足需求；保留末尾断言，不能只检查累计值。调试时可以自行增删打印。
- 将修复后的函数填入组装格的 `make_accumulator`，再运行单元检查。


In [ ]:
# 重新运行本格，恢复场景六的起点与增量。
q6_start = 10
q6_parent_steps = [2, 3]
q6_child_step = -5
print("起点：", q6_start)
print("先加 2，再保存快照并分叉；原路线加 3，新路线加 -5。")
print("预期原路线为 15、新路线为 7，旧快照仍为 12 且只有第一条历史。")

# 我的预测：两条路线各有哪些历史？修改旧快照中的字典后会怎样？


In [ ]:
# Playground：先预测，再运行；观察分支历史和嵌套快照的变化。
def make_accumulator_trial(start=0):
    def build(value, history):
        def add(step):
            nonlocal value
            before = value
            value += step
            history.append({"before": before, "step": step, "after": value})
            return value

        def snapshot():
            return {"value": value, "history": history}

        def fork():
            return build(value, history)

        return {"add": add, "snapshot": snapshot, "fork": fork}

    return build(start, [])


q6_parent_trial = make_accumulator_trial(q6_start)
q6_parent_trial["add"](q6_parent_steps[0])
q6_saved_trial = q6_parent_trial["snapshot"]()
q6_child_trial = q6_parent_trial["fork"]()
q6_parent_trial["add"](q6_parent_steps[1])
q6_child_trial["add"](q6_child_step)
print("原路线：", q6_parent_trial["snapshot"]())
print("新路线：", q6_child_trial["snapshot"]())
print("旧快照：", q6_saved_trial)
q6_other_snapshot_trial = q6_parent_trial["snapshot"]()
q6_saved_trial["history"][0]["after"] = -999
print("修改旧快照的第一条记录后，原路线：", q6_parent_trial["snapshot"]())
print("修改后，新路线：", q6_child_trial["snapshot"]())
print("修改后，另一份快照：", q6_other_snapshot_trial)

q6_expected_parent_trial = {"value": 15, "history": [
    {"before": 10, "step": 2, "after": 12},
    {"before": 12, "step": 3, "after": 15},
]}
q6_expected_child_trial = {"value": 7, "history": [
    {"before": 10, "step": 2, "after": 12},
    {"before": 12, "step": -5, "after": 7},
]}
assert q6_parent_trial["snapshot"]() == q6_expected_parent_trial, "原路线应只保留自己的历史"
assert q6_child_trial["snapshot"]() == q6_expected_child_trial, "新路线应只保留自己的历史"
assert q6_other_snapshot_trial == q6_expected_parent_trial, "其他快照中的嵌套记录应保持不变"
assert q6_saved_trial == {"value": 12, "history": [
    {"before": 10, "step": 2, "after": -999},
]}, "旧快照只应有保存时的历史，以及对它主动做出的编辑"


In [ ]:
# 正式接口；允许在函数内部组织辅助函数，不使用 class。
def make_accumulator(start=0):
    raise NotImplementedError("请组装有快照和独立分叉的累加器")


In [ ]:
q6_a = make_accumulator(10)
assert set(q6_a) == {"add", "snapshot", "fork"}
assert all(callable(action) for action in q6_a.values())
assert q6_a["snapshot"]() == {"value": 10, "history": []}
assert q6_a["add"](2) == 12
q6_saved = q6_a["snapshot"]()
q6_b = q6_a["fork"]()
assert q6_b["snapshot"]() == q6_saved
assert q6_a["add"](3) == 15
assert q6_b["add"](-5) == 7

q6_expected_a = {"value": 15, "history": [
    {"before": 10, "step": 2, "after": 12},
    {"before": 12, "step": 3, "after": 15},
]}
q6_expected_b = {"value": 7, "history": [
    {"before": 10, "step": 2, "after": 12},
    {"before": 12, "step": -5, "after": 7},
]}
assert q6_a["snapshot"]() == q6_expected_a
assert q6_b["snapshot"]() == q6_expected_b
assert q6_saved == {"value": 12, "history": [
    {"before": 10, "step": 2, "after": 12},
]}

q6_other_snapshot = q6_a["snapshot"]()
q6_saved["value"] = 999
q6_saved["history"][0]["after"] = 999
q6_saved["history"].append({"before": 0, "step": 0, "after": 0})
assert q6_a["snapshot"]() == q6_other_snapshot == q6_expected_a
assert q6_b["snapshot"]() == q6_expected_b

q6_c = q6_b["fork"]()
assert q6_c["add"](0) == 7
assert q6_c["snapshot"]()["history"][-1] == {"before": 7, "step": 0, "after": 7}
assert len(q6_c["snapshot"]()["history"]) == 3
assert q6_b["snapshot"]() == q6_expected_b
assert q6_a["snapshot"]() == q6_expected_a
assert make_accumulator(10)["snapshot"]() == {"value": 10, "history": []}
assert make_accumulator()["snapshot"]() == {"value": 0, "history": []}
q6_negative = make_accumulator(-3)
assert q6_negative["add"](-2) == -5
print("题 6：PASS（实例内共享、分支独立、嵌套快照隔离与重复分叉）")


<details>
<summary>参考解答（点击展开）</summary>

```python
from copy import deepcopy

def make_accumulator(start=0):
    def build(value, history):
        def add(step):
            nonlocal value
            before = value
            value += step
            history.append({"before": before, "step": step, "after": value})
            return value

        def snapshot():
            return {"value": value, "history": deepcopy(history)}

        def fork():
            return build(value, deepcopy(history))

        return {"add": add, "snapshot": snapshot, "fork": fork}

    return build(start, [])
```

#### 错误原因

`snapshot()` 把内部的 `history` 列表直接放进快照，`fork()` 又把同一个列表交给新分支。两条路线虽然各自更新自己的 `value`，却会往同一份历史里追加记录；旧快照的历史也会继续增长。历史中的字典同样被共用，所以修改旧快照的一条记录，会影响原路线、新路线和其他快照。

#### 修正前后的本质差别

- **修正前**：快照、原路线和新路线共用同一份历史，无法分别保留保存时的记录或各自的后续记录。
- **修正后**：同一次 `build` 产生的函数仍共享本实例的 `value` 与 `history`；导出快照和建立分支时，分别用 `deepcopy(history)` 复制历史。各分支与各份快照的列表、列表内的字典都相互独立，后续追加或修改记录不会波及其他对象。

</details>
